In [ ]:
import os

# ---- Chemin du dataset (CSV) ----
BASE = "/kaggle/input/datasets/souhilalpha/data-center-temperature-humidity-sensors"

# Cherche automatiquement le dossier qui contient les fichiers .csv (même s'il est dans un sous-dossier)
FOLDER = None
for root, dirs, files in os.walk(BASE):
    if "Temp_Cold_Rack_1.csv" in files:
        FOLDER = root
        break
assert FOLDER is not None, f"Aucun fichier Temp_Cold_Rack_1.csv trouvé sous {BASE}"

print("Dossier des données :", FOLDER)
for f in sorted(os.listdir(FOLDER)):
    if f.endswith(".csv"):
        print(os.path.join(FOLDER, f))


In [ ]:
import pandas as pd

folder = FOLDER   # défini dans la cellule précédente

names = ["Temp_Cold_Rack_1","Temp_Cold_Rack_2","Temp_Hot_Rack_1","Temp_Hot_Rack_2",
         "RH_Cold_Rack_1","RH_Cold_Rack_2","RH_Hot_Rack_1","RH_Hot_Rack_2"]

raw = {}
for n in names:
    d = pd.read_csv(f"{folder}/{n}.csv", encoding="utf-8-sig")   # CSV : séparateur virgule
    raw[n] = d
    print(f"\n=== {n}  shape={d.shape}")
    print("columns:", list(d.columns))

print("\nData types of one file:")
print(raw["Temp_Hot_Rack_1"].dtypes)
display(raw["Temp_Hot_Rack_1"].head(5))

In [ ]:
for n, d in raw.items():
    t = pd.to_datetime(d["Timestamp"])
    step = t.diff().dropna().dt.total_seconds()
    print(f"{n:18s} first={t.iloc[0]}  last={t.iloc[-1]}  "
          f"step(s) counts={step.value_counts().head(3).to_dict()}")

In [ ]:
import pandas as pd
import numpy as np

folder = FOLDER
names = ["Temp_Cold_Rack_1","Temp_Cold_Rack_2","Temp_Hot_Rack_1","Temp_Hot_Rack_2",
         "RH_Cold_Rack_1","RH_Cold_Rack_2","RH_Hot_Rack_1","RH_Hot_Rack_2"]

def load(n):
    d = pd.read_csv(f"{folder}/{n}.csv", encoding="utf-8-sig")
    d["Timestamp"] = pd.to_datetime(d["Timestamp"])
    d.columns = [c.replace("-Mild-", "-Mid-") for c in d.columns]   # unify naming
    d["t5"] = d["Timestamp"].dt.floor("5s")                          # align to 5-s grid
    return d.drop(columns="Timestamp").set_index("t5")

parts = {n: load(n) for n in names}

# Check 1: all files have exactly the same time index after alignment
ref = parts[names[0]].index
print("All indexes identical:", all(p.index.equals(ref) for p in parts.values()))
print("Index unique:", ref.is_unique, "| sorted:", ref.is_monotonic_increasing)

df = pd.concat(parts.values(), axis=1)
print("Merged shape:", df.shape)
print("Duplicate column names:", df.columns.duplicated().sum())
print("First / last time:", df.index[0], "/", df.index[-1])

temp_cols = [c for c in df.columns if c.endswith("-TEMP")]
rh_cols   = [c for c in df.columns if c.endswith("-RH")]
print("Temperature sensors:", len(temp_cols), "| Humidity sensors:", len(rh_cols))

df.to_csv("/kaggle/working/merged_raw.csv")   # saved so we don't redo this if the session restarts

In [ ]:
print("Total missing values:", int(df.isna().sum().sum()))
print("Duplicated rows (all values identical):", int(df.duplicated().sum()))
print("Duplicated timestamps:", int(df.index.duplicated().sum()))

# plausibility limits (physical, not data-driven)
bad_t  = ((df[temp_cols] < 0) | (df[temp_cols] > 60)).sum().sum()
bad_rh = ((df[rh_cols]  < 0) | (df[rh_cols]  > 100)).sum().sum()
print("Temperature readings outside 0-60 C:", int(bad_t))
print("Humidity readings outside 0-100 %:", int(bad_rh))

print("\n--- Temperature (C) per sensor ---")
display(df[temp_cols].describe().T[["min","mean","max","std"]].round(2))
print("\n--- Humidity (%) per sensor ---")
display(df[rh_cols].describe().T[["min","mean","max","std"]].round(2))

In [ ]:
def longest_run(s):
    g = (s != s.shift()).cumsum()
    return s.groupby(g).size().max()

runs = pd.Series({c: longest_run(df[c]) for c in df.columns})
print("Longest run of identical consecutive values (top 8):")
print(runs.sort_values(ascending=False).head(8))

jump = df.diff().abs().max()
print("\nLargest change in one 5-s step - temperature sensors (top 8):")
print(jump[temp_cols].sort_values(ascending=False).head(8).round(2))
print("\nLargest change in one 5-s step - humidity sensors (top 8):")
print(jump[rh_cols].sort_values(ascending=False).head(8).round(2))

In [ ]:
import os, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ---- CONFIG (change only these) ----
FOLDER = os.environ.get("TDC_FOLDER", FOLDER)      # FOLDER trouvé dans la cellule 0
OUT    = os.environ.get("TDC_OUT", "/kaggle/working")
SEED   = 42
np.random.seed(SEED); random.seed(SEED)
os.makedirs(OUT, exist_ok=True)
print("Folder:", FOLDER)
print("Files found:", sorted(os.listdir(FOLDER)))

In [ ]:
names = ["Temp_Cold_Rack_1","Temp_Cold_Rack_2","Temp_Hot_Rack_1","Temp_Hot_Rack_2",
         "RH_Cold_Rack_1","RH_Cold_Rack_2","RH_Hot_Rack_1","RH_Hot_Rack_2"]

def load(n):
    d = pd.read_csv(f"{FOLDER}/{n}.csv", encoding="utf-8-sig")
    d["Timestamp"] = pd.to_datetime(d["Timestamp"])
    d.columns = [c.replace("-Mild-", "-Mid-") for c in d.columns]
    d["t5"] = d["Timestamp"].dt.floor("5s")
    return d.drop(columns="Timestamp").set_index("t5")

parts = {n: load(n) for n in names}
ref = parts[names[0]].index
print("All indexes identical:", all(p.index.equals(ref) for p in parts.values()))
print("Index unique:", ref.is_unique, "| sorted:", ref.is_monotonic_increasing)

df = pd.concat(parts.values(), axis=1)
print("Merged shape:", df.shape, "| duplicate column names:", int(df.columns.duplicated().sum()))
print("Time range:", df.index[0], "->", df.index[-1])

temp_cols = [c for c in df.columns if c.endswith("-TEMP")]
rh_cols   = [c for c in df.columns if c.endswith("-RH")]
print("Temperature sensors:", len(temp_cols), "| Humidity sensors:", len(rh_cols))

# ---- quality checks ----
n_missing = int(df.isna().sum().sum())
print("\nMissing values:", n_missing)
print("Duplicated rows:", int(df.duplicated().sum()), "| duplicated timestamps:", int(df.index.duplicated().sum()))
print("Temperature outside 0-60 C:", int(((df[temp_cols] < 0) | (df[temp_cols] > 60)).sum().sum()))
print("Humidity outside 0-100 %:", int(((df[rh_cols] < 0) | (df[rh_cols] > 100)).sum().sum()))

print("\n--- Temperature (C) per sensor ---")
display(df[temp_cols].describe().T[["min","mean","max","std"]].round(2))
print("--- Humidity (%) per sensor ---")
display(df[rh_cols].describe().T[["min","mean","max","std"]].round(2))

def longest_run(s):
    g = (s != s.shift()).cumsum()
    return s.groupby(g).size().max()
runs = pd.Series({c: longest_run(df[c]) for c in df.columns})
print("\nLongest run of identical consecutive values (top 8):")
print(runs.sort_values(ascending=False).head(8))
jump = df.diff().abs().max()
print("\nLargest 5-s change, temperature (top 8):"); print(jump[temp_cols].sort_values(ascending=False).head(8).round(2))
print("Largest 5-s change, humidity (top 8):");    print(jump[rh_cols].sort_values(ascending=False).head(8).round(2))

# ---- cleaning: ONLY if something is missing (expected: nothing happens) ----
if n_missing > 0:
    before = len(df)
    df = df.interpolate(method="time", limit=3).dropna()   # fill gaps up to 15 s, drop the rest
    print(f"\nCleaning applied: rows {before} -> {len(df)}")
else:
    print("\nNo cleaning needed for missing values.")

# ---- RH glitches: isolated spikes far from the local median (window = 7 samples = 35 s) ----
med = df[rh_cols].rolling(7, center=True, min_periods=1).median()
glitch = (df[rh_cols] - med).abs() > 20
print("\nRH glitch readings flagged (|value - local median| > 20 points):", int(glitch.sum().sum()))
print(glitch.sum()[glitch.sum() > 0])
if glitch.any().any():
    df[rh_cols] = df[rh_cols].mask(glitch).interpolate(method="time", limit=3)
    df = df.dropna()
    print("-> glitches replaced by time interpolation. Rows kept:", len(df))
sat = (df[rh_cols] >= 99.99).sum()
print("RH readings stuck at the 100 % sensor limit (kept, but FLAGGED as unreliable):")
print(sat[sat > 0])

cold_t = [c for c in temp_cols if c.startswith("Cold")]; hot_t = [c for c in temp_cols if c.startswith("Hot")]
cold_rh = [c for c in rh_cols if c.startswith("Cold")];  hot_rh = [c for c in rh_cols if c.startswith("Hot")]
df.to_csv(f"{OUT}/merged_clean.csv")

In [ ]:
groups = {"Cold temp (C)": cold_t, "Hot temp (C)": hot_t, "Cold RH (%)": cold_rh, "Hot RH (%)": hot_rh}
rows = []
for g, cols in groups.items():
    v = df[cols].to_numpy()
    rows.append({"group": g, "min": v.min(), "max": v.max(), "mean": v.mean(), "std": v.std(),
                 "spread between sensor means": df[cols].mean().max() - df[cols].mean().min()})
display(pd.DataFrame(rows).set_index("group").round(2))

print("Per-sensor maximum, hot temperature (hottest first):")
print(df[hot_t].max().sort_values(ascending=False).round(2))

# ---- Graph 1: mean temperature over time
fig, ax = plt.subplots(figsize=(12,4))
df[cold_t].mean(axis=1).plot(ax=ax, label="Cold side (mean of 12 sensors)")
df[hot_t].mean(axis=1).plot(ax=ax, label="Hot side (mean of 12 sensors)")
df[hot_t].max(axis=1).plot(ax=ax, label="Hot side (hottest sensor)", alpha=.7)
ax.set_ylabel("Temperature (C)"); ax.set_xlabel("Time"); ax.set_title("Temperature over time"); ax.legend(); plt.show()

# ---- Graph 2: every hot-side sensor
fig, ax = plt.subplots(figsize=(12,4))
for c in hot_t: ax.plot(df.index, df[c], lw=.6, label=c.replace("-TEMP",""))
ax.set_ylabel("Temperature (C)"); ax.set_title("All hot-side sensors"); ax.legend(ncol=4, fontsize=7); plt.show()

# ---- Graph 3: humidity over time
fig, ax = plt.subplots(figsize=(12,4))
df[cold_rh].mean(axis=1).plot(ax=ax, label="Cold side mean RH")
df[hot_rh].mean(axis=1).plot(ax=ax, label="Hot side mean RH")
ax.set_ylabel("Relative humidity (%)"); ax.set_title("Humidity over time"); ax.legend(); plt.show()

# ---- Graphs 4-5: distributions
fig, axs = plt.subplots(1, 2, figsize=(12,4))
axs[0].hist(df[cold_t].to_numpy().ravel(), bins=50, alpha=.6, label="Cold")
axs[0].hist(df[hot_t].to_numpy().ravel(),  bins=50, alpha=.6, label="Hot")
axs[0].set_title("Temperature distribution"); axs[0].set_xlabel("C"); axs[0].legend()
axs[1].hist(df[cold_rh].to_numpy().ravel(), bins=50, alpha=.6, label="Cold")
axs[1].hist(df[hot_rh].to_numpy().ravel(),  bins=50, alpha=.6, label="Hot")
axs[1].set_title("Humidity distribution"); axs[1].set_xlabel("%"); axs[1].legend(); plt.show()

# ---- Graph 6: temperature vs humidity (subsampled to keep the plot readable)
sub = df.iloc[::10]
fig, axs = plt.subplots(1, 2, figsize=(12,4))
axs[0].scatter(sub[cold_t].mean(axis=1), sub[cold_rh].mean(axis=1), s=3); axs[0].set_title("Cold side")
axs[1].scatter(sub[hot_t].mean(axis=1),  sub[hot_rh].mean(axis=1),  s=3); axs[1].set_title("Hot side")
for a in axs: a.set_xlabel("Mean temperature (C)"); a.set_ylabel("Mean RH (%)")
plt.show()
print("Correlation temp vs RH, cold side:", round(df[cold_t].mean(axis=1).corr(df[cold_rh].mean(axis=1)), 3))
print("Correlation temp vs RH, hot side :", round(df[hot_t].mean(axis=1).corr(df[hot_rh].mean(axis=1)), 3))

# ---- thermal peaks and possible abnormal measurements
hot_max_s = df[hot_t].max(axis=1)
peaks = pd.DataFrame({"hot_max_C": hot_max_s, "sensor": df[hot_t].idxmax(axis=1)})
print("\nTop-10 hottest moments on the hot side:"); display(peaks.sort_values("hot_max_C", ascending=False).head(10))
q1, q3 = df[temp_cols].quantile(.25), df[temp_cols].quantile(.75); iqr = q3 - q1
out_cnt = ((df[temp_cols] < q1 - 3*iqr) | (df[temp_cols] > q3 + 3*iqr)).sum()
print("Readings beyond 3xIQR per temperature sensor (only those > 0):"); print(out_cnt[out_cnt > 0])

In [ ]:
HEIGHTS  = ["High", "Mid", "Low"]                       # top -> bottom
RACKS    = ["R1", "R2", "R3", "R4"]                     # left -> right
CHANNELS = [("Cold","TEMP"), ("Hot","TEMP"), ("Cold","RH"), ("Hot","RH")]
CH_NAMES = ["Cold TEMP", "Hot TEMP", "Cold RH", "Hot RH"]

def map_cols(side, kind):                               # row-major: height first, then rack
    return [f"{side}-{r}-{h}-{kind}" for h in HEIGHTS for r in RACKS]

needed = sum((map_cols(s, k) for s, k in CHANNELS), [])
missing = [c for c in needed if c not in df.columns]
assert not missing, f"Missing sensor columns: {missing}"

X_raw = np.stack([df[map_cols(s, k)].to_numpy().reshape(-1, 3, 4) for s, k in CHANNELS], axis=-1)
print("X_raw shape (samples, height, rack, channels):", X_raw.shape)

# sanity check: one cell of the array equals the original column
i0 = 1234
assert X_raw[i0, 2, 1, 1] == df["Hot-R2-Low-TEMP"].iloc[i0]
print("Check OK: X_raw[i,Low,R2,Hot TEMP] == Hot-R2-Low-TEMP")

# example maps: coolest, median and hottest moment on the hot side
hm = df[hot_t].max(axis=1).to_numpy()
order = np.argsort(hm)
examples = {"Coolest moment": order[0], "Median moment": order[len(order)//2], "Hottest moment": order[-1]}
vmin, vmax = X_raw[..., :2].min(), X_raw[..., :2].max()
fig, axs = plt.subplots(3, 2, figsize=(8, 9))
for r, (title, i) in enumerate(examples.items()):
    for c, side in enumerate(["Cold", "Hot"]):
        a = axs[r, c]; M = X_raw[i, :, :, c]
        im = a.imshow(M, cmap="inferno", vmin=vmin, vmax=vmax)
        a.set_xticks(range(4)); a.set_xticklabels(RACKS); a.set_yticks(range(3)); a.set_yticklabels(HEIGHTS)
        for y in range(3):
            for x in range(4): a.text(x, y, f"{M[y,x]:.1f}", ha="center", va="center", color="w", fontsize=9)
        a.set_title(f"{side} side - {title}\n{df.index[i]}", fontsize=9)
fig.colorbar(im, ax=axs, shrink=.6, label="Temperature (C)"); plt.show()

In [ ]:
N = len(df); GAP = 120                                  # 120 x 5 s = 10 minutes
i_tr_end = int(0.70 * N); i_va_s = i_tr_end + GAP; i_va_e = int(0.85 * N); i_te_s = i_va_e + GAP
idx_tr = np.arange(0, i_tr_end); idx_va = np.arange(i_va_s, i_va_e); idx_te = np.arange(i_te_s, N)
print("Train:", df.index[idx_tr[0]], "->", df.index[idx_tr[-1]], len(idx_tr))
print("Val  :", df.index[idx_va[0]], "->", df.index[idx_va[-1]], len(idx_va))
print("Test :", df.index[idx_te[0]], "->", df.index[idx_te[-1]], len(idx_te))
assert idx_tr[-1] < idx_va[0] < idx_va[-1] < idx_te[0]

# ---- labels (thresholds from TRAIN only) ----
FIXED_THRESHOLDS = None            # e.g. (32.0, 35.0) if you can justify physical values
if FIXED_THRESHOLDS is None:
    T_WARM, T_HOT = np.quantile(hm[idx_tr], [0.60, 0.90])
else:
    T_WARM, T_HOT = FIXED_THRESHOLDS
print(f"\nThresholds: Warm >= {T_WARM:.2f} C, Hotspot >= {T_HOT:.2f} C (on hottest hot-side sensor)")

CLASS_NAMES = ["Normal", "Warm", "Hotspot"]
y_all = np.digitize(hm, [T_WARM, T_HOT])               # 0,1,2
y_train, y_val, y_test = y_all[idx_tr], y_all[idx_va], y_all[idx_te]

share = df[hot_t].idxmax(axis=1).value_counts(normalize=True).round(3)
print("\nWhich hot-side sensor is the hottest? (share of time)"); print(share.to_dict())
if share.iloc[0] > 0.9:
    print(f"NOTE: {share.index[0]} is the hottest sensor {share.iloc[0]*100:.0f}% of the time -> the hotspot is spatially FIXED in this recording.")
    print("      The label is then effectively a threshold on one sensor. State this limitation in your report.")

bal = pd.DataFrame({s: np.bincount(y, minlength=3) for s, y in
                    [("train", y_train), ("val", y_val), ("test", y_test)]}, index=CLASS_NAMES)
print("\nClass balance (counts):"); display(bal)
print("Class balance (%):");       display((bal / bal.sum() * 100).round(1))
if (bal == 0).any().any():
    print("WARNING: some class is absent from a split -> metrics for it are undefined. Send me this output.")

plt.figure(figsize=(8,3.5)); plt.hist(hm, bins=60)
plt.axvline(T_WARM, color="orange", label="Warm"); plt.axvline(T_HOT, color="red", label="Hotspot")
plt.xlabel("hot_max (C)"); plt.title("Hottest hot-side sensor with label thresholds"); plt.legend(); plt.show()

# ---- normalization: fit on TRAIN only, per channel ----
mean_c = X_raw[idx_tr].mean(axis=(0, 1, 2)); std_c = X_raw[idx_tr].std(axis=(0, 1, 2)) + 1e-8
X = ((X_raw - mean_c) / std_c).astype("float32")
X_train, X_val, X_test = X[idx_tr], X[idx_va], X[idx_te]
print("\nCNN input shape = (samples, height, width, channels)")
print("X_train", X_train.shape, "| X_val", X_val.shape, "| X_test", X_test.shape)
print("channel means (train):", dict(zip(CH_NAMES, mean_c.round(2))))
print("train mean after scaling ~0:", X_train.mean(axis=(0,1,2)).round(3))

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(SEED)

def build_cnn(input_shape=(3, 4, 4), n_classes=3, dropout=0.0):
    return keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(16, (3, 3), padding="same", activation="relu"),   # local patterns, 16 filters
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),   # combinations of patterns
        layers.MaxPooling2D((2, 2)),                                    # 3x4 -> 1x2
        layers.Flatten(),
        layers.Dense(32, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(n_classes, activation="softmax"),
    ])

model_base = build_cnn(dropout=0.0)
model_base.summary()

In [ ]:
model_base.compile(optimizer=keras.optimizers.Adam(1e-3),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
hist_base = model_base.fit(X_train, y_train, validation_data=(X_val, y_val),
                           epochs=60, batch_size=64, callbacks=[early], verbose=2)
print("Stopped after", len(hist_base.history["loss"]), "epochs")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

def evaluate_on(model, X_, y_):
    p = model.predict(X_, verbose=0); return p, p.argmax(axis=1)

def curves(h, title):
    fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
    axs[0].plot(h.history["loss"], label="train"); axs[0].plot(h.history["val_loss"], label="val"); axs[0].set_title(title + " - loss"); axs[0].legend()
    axs[1].plot(h.history["accuracy"], label="train"); axs[1].plot(h.history["val_accuracy"], label="val"); axs[1].set_title(title + " - accuracy"); axs[1].legend()
    plt.show()

def full_report(model, h, title):
    curves(h, title)
    p, pred = evaluate_on(model, X_test, y_test)
    print(f"[{title}] TEST accuracy:", round(accuracy_score(y_test, pred), 4))
    maj = np.bincount(y_train).argmax()
    print("Majority-class baseline accuracy:", round(float((y_test == maj).mean()), 4))
    print(classification_report(y_test, pred, labels=[0,1,2], target_names=CLASS_NAMES, zero_division=0))
    cm = confusion_matrix(y_test, pred, labels=[0,1,2])
    fig, ax = plt.subplots(figsize=(4.5, 4)); ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(3)); ax.set_xticklabels(CLASS_NAMES); ax.set_yticks(range(3)); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion matrix (test)")
    for i in range(3):
        for j in range(3): ax.text(j, i, cm[i, j], ha="center", va="center", color="red")
    plt.show()
    # where do errors happen? distance of hot_max to the nearest threshold
    hm_te = hm[idx_te]; wrong = pred != y_test
    dist = np.minimum(abs(hm_te - T_WARM), abs(hm_te - T_HOT))
    if wrong.any():
        print(f"Errors: {int(wrong.sum())}. Median distance of wrong maps to nearest threshold: {np.median(dist[wrong]):.2f} C "
              f"(vs {np.median(dist[~wrong]):.2f} C for correct ones)")
    return p, pred

p_te, pred_te = full_report(model_base, hist_base, "Base CNN")

rng = np.random.default_rng(SEED); pick = np.sort(rng.choice(len(y_test), 12, replace=False))
ex = pd.DataFrame({"time": df.index[idx_te][pick], "hot_max_C": hm[idx_te][pick].round(2),
                   "actual": [CLASS_NAMES[k] for k in y_test[pick]],
                   "predicted": [CLASS_NAMES[k] for k in pred_te[pick]],
                   "P(hotspot)": p_te[pick, 2].round(3)})
ex["correct"] = ex["actual"] == ex["predicted"]
print("Actual vs predicted (random test examples):"); display(ex)

fig, axs = plt.subplots(2, 3, figsize=(10, 6))
for a, k in zip(axs.ravel(), pick[:6]):
    M = X_raw[idx_te[k], :, :, 1]; a.imshow(M, cmap="inferno", vmin=vmin, vmax=vmax)
    a.set_xticks(range(4)); a.set_xticklabels(RACKS); a.set_yticks(range(3)); a.set_yticklabels(HEIGHTS)
    a.set_title(f"actual {CLASS_NAMES[y_test[k]]} / pred {CLASS_NAMES[pred_te[k]]}", fontsize=9)
plt.suptitle("Hot-side maps from the test set"); plt.show()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

def diagnose(h, name):
    tl, vl = h.history["loss"], h.history["val_loss"]; ta, va = h.history["accuracy"], h.history["val_accuracy"]
    best = int(np.argmin(vl))
    print(f"[{name}] epochs={len(tl)} best_epoch={best+1} | train acc={ta[best]:.3f} val acc={va[best]:.3f} gap={ta[best]-va[best]:+.3f}")
    if ta[best] - va[best] > 0.05: print("   -> train >> val: possible overfitting")
    if va[best] < 0.80:            print("   -> low accuracy: check labels/maps or try a bigger model")
diagnose(hist_base, "Base")

present = np.unique(y_train)
w = compute_class_weight("balanced", classes=present, y=y_train)
class_w = {int(c): float(x) for c, x in zip(present, w)}
print("Class weights:", class_w)

keras.utils.set_random_seed(SEED)
model_imp = build_cnn(dropout=0.3)
model_imp.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist_imp = model_imp.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=60, batch_size=64,
                         class_weight=class_w, callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)], verbose=0)
diagnose(hist_imp, "Improved")

def macro_f1(model, X_, y_):
    _, pr = evaluate_on(model, X_, y_); return f1_score(y_, pr, average="macro", labels=np.unique(y_), zero_division=0)
cmp = pd.DataFrame({"val macro-F1": [macro_f1(model_base, X_val, y_val), macro_f1(model_imp, X_val, y_val)],
                    "test macro-F1": [macro_f1(model_base, X_test, y_test), macro_f1(model_imp, X_test, y_test)]},
                   index=["Base", "Improved (dropout+weights)"]).round(4)
display(cmp)
best_model, best_hist, best_name = (model_imp, hist_imp, "Improved") if cmp["val macro-F1"].iloc[1] > cmp["val macro-F1"].iloc[0] else (model_base, hist_base, "Base")
print("Selected by VALIDATION macro-F1:", best_name)
if best_name == "Improved": full_report(best_model, best_hist, "Improved CNN")

In [ ]:
P = best_model.predict(X, verbose=0)
hot_arr = df[hot_t].to_numpy(); arg = hot_arr.argmax(axis=1)
split = np.full(N, "gap", dtype=object); split[idx_tr] = "train"; split[idx_va] = "val"; split[idx_te] = "test"

out = pd.DataFrame({
    "split": split,
    "thermal_state_code": P.argmax(axis=1),
    "thermal_state": [CLASS_NAMES[k] for k in P.argmax(axis=1)],
    "p_normal": P[:, 0], "p_warm": P[:, 1], "p_hotspot": P[:, 2],
    "hot_max_C": hot_arr.max(axis=1), "hot_mean_C": hot_arr.mean(axis=1),
    "cold_mean_C": df[cold_t].mean(axis=1).to_numpy(),
    "hottest_rack":   [hot_t[i].split("-")[1] for i in arg],
    "hottest_height": [hot_t[i].split("-")[2] for i in arg],
    "rh_hot_mean": df[hot_rh].mean(axis=1).to_numpy(), "rh_cold_mean": df[cold_rh].mean(axis=1).to_numpy(),
}, index=df.index)
out["delta_T_C"] = out["hot_mean_C"] - out["cold_mean_C"]
out.index.name = "timestamp"
out.to_csv(f"{OUT}/cnn_output_5s.csv")

agg = out.drop(columns=["split", "thermal_state", "hottest_rack", "hottest_height"]).resample("1min").agg(
      {"p_normal":"mean","p_warm":"mean","p_hotspot":"mean","hot_max_C":"max","hot_mean_C":"mean",
       "cold_mean_C":"mean","rh_hot_mean":"mean","rh_cold_mean":"mean","delta_T_C":"mean","thermal_state_code":"max"})
agg.to_csv(f"{OUT}/cnn_output_1min.csv")

best_model.save(f"{OUT}/cnn_thermal.keras")
meta = {"classes": CLASS_NAMES, "thresholds_C": {"warm": float(T_WARM), "hotspot": float(T_HOT)},
        "label_rule": "hottest hot-side sensor (hot_max)", "channels": CH_NAMES, "map_shape_hw": [3, 4],
        "height_order_top_to_bottom": HEIGHTS, "rack_order_left_to_right": RACKS,
        "normalization_mean": mean_c.tolist(), "normalization_std": std_c.tolist(),
        "split": {"train": [str(df.index[idx_tr[0]]), str(df.index[idx_tr[-1]])],
                  "val": [str(df.index[idx_va[0]]), str(df.index[idx_va[-1]])],
                  "test": [str(df.index[idx_te[0]]), str(df.index[idx_te[-1]])]},
        "selected_model": best_name, "seed": SEED}
json.dump(meta, open(f"{OUT}/cnn_handoff_meta.json", "w"), indent=2)

print("Saved:", sorted(f for f in os.listdir(OUT) if f.startswith(("cnn_", "merged"))))
display(out.head(5)); display(out[out.split == "test"].thermal_state.value_counts())